In [4]:
%pip install datasets matplotlib scikit-learn
from datasets import load_dataset

ds = load_dataset("rajistics/indian_food_images")


Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


In [6]:
ds=ds["train"].train_test_split(test_size=0.2,seed=42)
val_test=ds["test"].train_test_split(test_size=0.5, seed=42)

train_ds=ds["train"]
val_ds=val_test["train"]
test_ds=val_test["test"]

In [7]:
IMG_SIZE = 224
BATCH_SIZE=32

def preprocess(example):
    image=example["image"]
    image=image.resize((IMG_SIZE, IMG_SIZE))
    image=np.array(image, stype=np.float32)/255.0
    label=example["label"]
    return image, label


In [ ]:
def to_tf_dataset(hf_ds, shuffle=True, is_training=True):
    def generator():
        for example in hf_ds:
            img = example["image"]
            # Ensure image is in RGB (converts RGBA or Grayscale automatically)
            img = img.convert("RGB") 
            yield np.array(img), example["label"]

    tf_ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            # We use None for height/width because images might be different sizes
            tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8), 
            tf.TensorSpec(shape=(), dtype=tf.int64),
        ),
    )

    def preprocess(image, label):
        # Resize and scale
        image = tf.image.resize(image, (224, 224))
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    tf_ds = tf_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        tf_ds = tf_ds.shuffle(1000)
    
    if is_training:
        tf_ds = tf_ds.repeat()

    return tf_ds.batch(32).prefetch(tf.data.AUTOTUNE)

# Re-initialize your datasets
train_tf = to_tf_dataset(train_ds, shuffle=True, is_training=True)
val_tf   = to_tf_dataset(val_ds, shuffle=False, is_training=False) # No repeat
test_tf  = to_tf_dataset(test_ds, shuffle=False, is_training=False) # No repeat

# Calculate steps accurately so the progress bar moves
steps_per_epoch = len(train_ds) // 32
val_steps = len(val_ds) // 32

2026-02-14 13:18:33.136285: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-02-14 13:18:33.136318: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-02-14 13:18:33.136325: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-02-14 13:18:33.136355: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-02-14 13:18:33.136364: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [10]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2)
])


In [11]:
base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False


In [12]:
num_classes = len(set(train_ds["label"]))


In [13]:
inputs = tf.keras.Input(shape=(224,224,3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.5)(x)

outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)


In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [15]:
history = model.fit(
    train_tf,
    validation_data=val_tf,
    epochs=15,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps
)


Epoch 1/15


2026-02-14 13:18:34.638977: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


133/133 ━━━━━━━━━━━━━━━━━━━━ 65s 409ms/step - accuracy: 0.1074 - loss: 3.5232 - val_accuracy: 0.3281 - val_loss: 2.3927
Epoch 2/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 53s 403ms/step - accuracy: 0.2234 - loss: 2.8906 - val_accuracy: 0.5098 - val_loss: 1.9011
Epoch 3/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 54s 405ms/step - accuracy: 0.3228 - loss: 2.4805 - val_accuracy: 0.5879 - val_loss: 1.5674
Epoch 4/15
133/133 ━━━━━━━━━━━━━━━━━━━━ 53s 402ms/step - accuracy: 0.3983 - loss: 2.1147 - val_accuracy: 0.6582 - val_loss: 1.3387
Epoch 5/15
122/133 ━━━━━━━━━━━━━━━━━━━━ 3s 356ms/step - accuracy: 0.4405 - loss: 1.9836

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import classification_report

y_true = []
y_pred = []

for images, labels in test_tf:
    preds = model.predict(images)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print(classification_report(y_true, y_pred))


In [ ]:
model.save("indian_food_classifier.h5")


In [ ]:
import json
from datasets import load_dataset

dataset = load_dataset("rajistics/indian_food_images")

# Get label names
label_names = dataset["train"].features["label"].names

# Save to json
with open("label_map.json", "w") as f:
    json.dump(label_names, f)

print("label_map.json created ✅")
